# 01 — Baseline spike (issue #2)

The highest-risk unknown in the whole project, deliberately checked first: does our own harness reproduce Stanford's published CLMBR number? If this doesn't match, everything built on top of these features is suspect.

**Already verified against real data locally (Sep 20)**: AUROC 0.8153 vs. published 0.81374 for `guo_los` (Long Length of Stay) — reproduced, within normal tolerance. This notebook re-runs the same check in Colab so it's on record there too, and is the template for checking Hypertension (Task B) the same way.

In [ ]:
import os
import sys

from google.colab import userdata

token = userdata.get("GH_PAT")

BRANCH = "main"

REPO_DIR = "ehrshot-patient-graphs"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout {BRANCH}
    !git reset --hard origin/{BRANCH}
else:
    !git clone --branch {BRANCH} https://{token}@github.com/patient-graph-lab/ehrshot-patient-graphs.git
    os.chdir(REPO_DIR)

!pip install -e ".[dev,data]"

src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
from scripts.colab_setup import ensure_ehrshot_data

data_root = ensure_ehrshot_data(checksums_path="data/checksums.txt")

In [ ]:
from ehrgraph.eval.baseline import run_baseline_spike

assets = data_root / "EHRSHOT_ASSETS"

result = run_baseline_spike(
    features_path=assets / "features" / "clmbr_features.pkl",
    labeled_patients_path=assets / "benchmark" / "guo_los" / "labeled_patients.csv",
    person_id_map_path=assets / "splits" / "person_id_map.csv",
)
print(result)

# EHRSHOT_ASSETS/results/guo_los/all_results.csv, model=clmbr, head=lr_lbfgs, k=-1
PUBLISHED_AUROC = 0.81374
diff = abs(result["auroc"] - PUBLISHED_AUROC)
print(f"published={PUBLISHED_AUROC}  diff={diff:.4f}")
assert diff < 0.02, "Baseline did not reproduce -- stop and investigate before anything else"

## Same check for Hypertension (Task B)

In [ ]:
import pandas as pd

results_b = pd.read_csv(assets / "results" / "new_hypertension" / "all_results.csv")
published_b = results_b[
    (results_b["model"] == "clmbr")
    & (results_b["head"] == "lr_lbfgs")
    & (results_b["k"] == -1)
    & (results_b["score"] == "auroc")
]["value"]
print("published (new_hypertension, clmbr, lr_lbfgs, k=-1):", published_b.to_list())

In [ ]:
result_b = run_baseline_spike(
    features_path=assets / "features" / "clmbr_features.pkl",
    labeled_patients_path=assets / "benchmark" / "new_hypertension" / "labeled_patients.csv",
    person_id_map_path=assets / "splits" / "person_id_map.csv",
)
print(result_b)